# 04c Team 3 re-implementation: correction ladder

**Purpose.** Re-implement Team 3's hybrid NCF + LSTM model (the prior project baseline) and correct its flaws **one at a time**, measuring each step on our protocol (complete validation lists, NDCG@10 with linear gains, exact tie expectation). This shows how much each diagnosed flaw costs.

| Rung | Change (cumulative) | Flaw addressed |
|---|---|---|
| **R0** | Team 3 as published: sampled training lists, history filtered by the sampled lists **of all users** (label-dependent, includes evaluation labels) without the gap to the search, history-vocabulary candidate lookup (attributes misaligned), weighted cross-entropy, ranked by the **maximum class probability** | - |
| **R0b** | the same filter built from **training lists only** | label leak through the catalogue cut |
| **R1** | consistent candidate vocabulary | wrong product lookup |
| **R2** | rank by **expected label** `sum_k k * p_k` | ranking by a confident "no interaction" |
| **R3** | full history, gaps including the gap to the search, corrected time decay, days since the last action | history gutted, time gaps corrupted (and the label leak through filtering) |
| **R4** | train on **complete** candidate lists | downsampled training lists |
| **R5** | add our non-personalised candidate features (relevance, popularity; notebook 03) | no query / candidate-level relevance input |

**Model, as in Team 3's `Models/3.1_Hybrid_NCF_with_LSTM.ipynb`:** item + type + time-bin embeddings (64) into a 2-layer LSTM with attention (empty histories get a learned vector); candidate encoder summing product, brand, shop and L1-L4 embeddings; 15 user meta features through a linear layer; concatenation into an MLP (128, 64, BatchNorm, dropout 0.3) with 4 classes; class-weighted cross-entropy; Adam (lr 1e-3, weight decay 1e-5), ReduceLROnPlateau (factor 0.5, patience 2), batches of 32 searches, at most 10 epochs, best epoch chosen on an inner validation split of the training searches (stopping after 3 epochs without improvement).

**Inputs:** the files of notebook 04b (working folder, `processed/team3/`). **Outputs:** `ladder_summary.csv` and `ladder_per_search.parquet` in the same folder; every rung in `results.csv`.

**Runtime:** T4 GPU, about 1.5 hours for all seven rungs. Without a GPU the notebook switches to a **smoke test** (2,000 training and 1,000 validation searches, one epoch) that only checks the pipeline; its numbers are not results.

In [1]:
# torch first: on Windows, importing numpy / pandas / scikit-learn before torch can break torch's DLL loading.
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence

# Setup: on Colab, clone the repository (shared code in jdsearch.py) and use Google Drive as the working folder.
import subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/repo").is_dir():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/liamkunert/BT4222-group17-jdsearch-submission.git", "/content/repo"], check=True)
    sys.path.insert(0, "/content/repo")
else:
    sys.path.insert(0, str(Path.cwd().resolve().parent))
import jdsearch as jd

paths = jd.setup()
jd.require(paths, "processed/team3/train_dev.npz", "processed/team3/validation.npz", "processed/team3/lookups.npz",
           "processed/team3/meta.json", "processed/search_table.parquet")

Mounted at /content/drive
working folder: /content/drive/MyDrive/BT4222-G17-JDsearch


In [2]:
import json
import time

import numpy as np
import pandas as pd

ARTIFACTS = paths.processed / "team3"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE = DEVICE.type != "cuda"  # smoke test without a GPU: small subsets, one epoch
MAX_EPOCHS, PATIENCE, BATCH = (1, 1, 32) if SMOKE else (10, 3, 32)
SMOKE_TRAIN, SMOKE_VAL = 2000, 1000
SEEDS = [1]
info = json.loads((ARTIFACTS / "meta.json").read_text())
print(f"device {DEVICE}; smoke test: {SMOKE}")

device cuda; smoke test: False


## 1. Load the prepared arrays

In [3]:
def load(split):
    data = dict(np.load(ARTIFACTS / f"{split}.npz"))
    n = len(data["session_id"])
    keep = np.arange(n)
    if SMOKE:
        keep = np.sort(np.random.default_rng(0).choice(n, min(n, SMOKE_TRAIN if split == "train_dev" else SMOKE_VAL),
                                                       replace=False))
    return data, keep


train_data, train_keep = load("train_dev")
val_data, val_keep = load("validation")
lookup = np.load(ARTIFACTS / "lookups.npz")["lookup"]
print(f"training searches {len(train_keep):,}, validation searches {len(val_keep):,}")

training searches 40,000, validation searches 20,000


## 2. Batching

In [4]:
def search_slice(data, i, key):
    off = data[f"{key}_offsets"]
    return slice(off[i], off[i + 1])


def make_batch(data, rows, cfg, train_lists):
    hist_key = f"hist_{cfg['history']}"
    items, types, times, lengths, cands, labels, metas, feats = [], [], [], [], [], [], [], []
    for i in rows:
        h = search_slice(data, i, hist_key)
        items.append(torch.as_tensor(data[f"{hist_key}_item"][h], dtype=torch.long))
        types.append(torch.as_tensor(data[f"{hist_key}_type"][h], dtype=torch.long))
        times.append(torch.as_tensor(data[f"{hist_key}_time"][h], dtype=torch.long))
        lengths.append(h.stop - h.start)
        c = search_slice(data, i, "cand")
        use = np.ones(c.stop - c.start, bool) if train_lists == "full" else data["sampled"][c].astype(bool)
        bug_key = "cand_bug_idx" if cfg["history"] == "filtered" else "cand_bug_idx_train"
        idx = data[bug_key if cfg["lookup"] == "bug" else "cand_idx"][c][use]
        cands.append(torch.as_tensor(idx, dtype=torch.long))
        labels.append(torch.as_tensor(data["label"][c][use], dtype=torch.long))
        metas.append(torch.as_tensor(data[f"meta_{cfg['history']}"][i]))
        feats.append(torch.as_tensor(data["r5_features"][c][use]))
    pad = lambda xs, v: pad_sequence(xs, batch_first=True, padding_value=v)
    empty = torch.zeros(0, dtype=torch.long)
    items = [x if len(x) else empty for x in items]
    return {"items": pad(items, 0) if max(lengths) else torch.zeros(len(rows), 1, dtype=torch.long),
            "types": pad(types, 0) if max(lengths) else torch.zeros(len(rows), 1, dtype=torch.long),
            "times": pad(times, 0) if max(lengths) else torch.zeros(len(rows), 1, dtype=torch.long),
            "lengths": torch.tensor(lengths), "cands": pad(cands, 0), "labels": pad(labels, -1),
            "meta": torch.stack(metas), "feats": pad(feats, 0.0)}

## 3. Team 3's model

In [5]:
class Attention(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.attn = nn.Linear(emb_dim, 1)

    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1).masked_fill(mask == 0, float("-inf"))
        return (x * torch.softmax(scores, dim=1).unsqueeze(-1)).sum(dim=1)


class CandidateEncoder(nn.Module):
    # Sum of product, brand, shop and L1-L4 embeddings; attributes looked up by candidate index.
    def __init__(self, num_wid, num_attr, lookup, emb_dim=64):
        super().__init__()
        self.wid_emb = nn.Embedding(num_wid, emb_dim, padding_idx=0)
        self.attr_emb = nn.ModuleList([nn.Embedding(n, emb_dim, padding_idx=0) for n in num_attr])
        padded = np.zeros((lookup.shape[0], num_wid), np.int64)  # indices beyond the table: unknown attributes
        padded[:, :lookup.shape[1]] = lookup
        self.register_buffer("lookup", torch.as_tensor(padded))

    def forward(self, idx):
        out = self.wid_emb(idx)
        for j, emb in enumerate(self.attr_emb):
            out = out + emb(self.lookup[j][idx])
        return out


class SequentialHybridNCF(nn.Module):
    def __init__(self, num_items, num_types, num_time_bins, num_wid, num_attr, lookup, meta_dim, extra_dim=0,
                 emb_dim=64, hidden_dim=128, num_classes=4):
        super().__init__()
        self.item_emb = nn.Embedding(num_items, emb_dim, padding_idx=0)
        self.type_emb = nn.Embedding(num_types, emb_dim, padding_idx=0)
        self.time_emb = nn.Embedding(num_time_bins, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, emb_dim, batch_first=True, dropout=0.3, num_layers=2)
        self.attention = Attention(emb_dim)
        self.empty_user_emb = nn.Parameter(torch.zeros(1, emb_dim))
        self.candidate_encoder = CandidateEncoder(num_wid, num_attr, lookup, emb_dim)
        self.meta_linear = nn.Linear(meta_dim, emb_dim)
        self.extra_linear = nn.Linear(extra_dim, emb_dim) if extra_dim else None
        fusion_dim = emb_dim * (4 if extra_dim else 3)
        self.mlp = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim), nn.ReLU(), nn.BatchNorm1d(hidden_dim), nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.BatchNorm1d(hidden_dim // 2), nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, num_classes))

    def encode_user(self, items, types, times, lengths):
        x = self.item_emb(items) + self.type_emb(types) + self.time_emb(times)
        out = self.empty_user_emb.repeat(x.size(0), 1)
        has = lengths > 0
        if has.any():
            packed = pack_padded_sequence(x[has], lengths[has].cpu(), batch_first=True, enforce_sorted=False)
            seq, _ = pad_packed_sequence(self.lstm(packed)[0], batch_first=True)
            mask = (items[has][:, :seq.size(1)] != 0) | (torch.arange(seq.size(1), device=seq.device)[None]
                                                         < lengths[has][:, None].to(seq.device))
            out[has] = self.attention(seq, mask)
        return out

    def forward(self, b):
        n = b["cands"].size(1)
        user = self.encode_user(b["items"], b["types"], b["times"], b["lengths"]).unsqueeze(1).expand(-1, n, -1)
        parts = [user, self.candidate_encoder(b["cands"]), self.meta_linear(b["meta"]).unsqueeze(1).expand(-1, n, -1)]
        if self.extra_linear is not None:
            parts.append(self.extra_linear(b["feats"]))
        fusion = torch.cat(parts, dim=-1)
        # As in Team 3: the MLP (with BatchNorm) runs on all positions, padding included.
        return self.mlp(fusion.reshape(-1, fusion.size(-1))).reshape(fusion.size(0), n, -1)

## 4. Training and scoring

In [6]:
def to_device(b):
    return {k: v.to(DEVICE) for k, v in b.items()}


def class_weights(data, rows, train_lists):
    counts = np.zeros(4)
    for i in rows:
        c = search_slice(data, i, "cand")
        labels = data["label"][c] if train_lists == "full" else data["label"][c][data["sampled"][c].astype(bool)]
        counts += np.bincount(labels, minlength=4)
    w = 1.0 / np.maximum(counts, 1)
    return torch.tensor(w / w.sum(), dtype=torch.float32, device=DEVICE)


def run_epoch(model, rows, cfg, criterion, optimizer=None):
    model.train(optimizer is not None)
    total, count = 0.0, 0
    order = np.random.permutation(rows) if optimizer is not None else rows
    for start in range(0, len(order), BATCH):
        b = to_device(make_batch(train_data, order[start:start + BATCH], cfg, cfg["train_lists"]))
        with torch.set_grad_enabled(optimizer is not None):
            logits = model(b)
            mask = b["labels"] != -1
            loss = criterion(logits[mask], b["labels"][mask])
            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total += loss.item() * mask.sum().item()
        count += mask.sum().item()
    return total / count


def train_rung(cfg, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    rows = train_keep
    inner = rows[train_data["inner_val"][rows]]
    fit = rows[~train_data["inner_val"][rows]]
    model = SequentialHybridNCF(
        num_items=info[f"num_hist_{cfg['history']}"], num_types=info["num_types"],
        num_time_bins=info["num_time_bins"],
        num_wid=max(info["num_cand"], info["num_hist_filtered"], info["num_hist_filtered_train"]),
        num_attr=info["num_attr"], lookup=lookup, meta_dim=info["meta_dims"][cfg["history"]],
        extra_dim=len(info["r5_features"]) if cfg["extra"] else 0).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights(train_data, fit, cfg["train_lists"]))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    best, best_state, waited = np.inf, None, 0
    for epoch in range(MAX_EPOCHS):
        start = time.time()
        train_loss = run_epoch(model, fit, cfg, criterion, optimizer)
        val_loss = run_epoch(model, inner, cfg, criterion)
        scheduler.step(val_loss)
        print(f"  epoch {epoch + 1}: train {train_loss:.4f}, inner val {val_loss:.4f}, {time.time() - start:.0f} s")
        if val_loss < best:
            best, waited = val_loss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model


@torch.no_grad()
def score_validation(model, cfg, lists="full"):
    # Scores for every validation candidate (complete lists, or Team 3's sampled lists).
    model.eval()
    sessions, labels, scores = [], [], []
    for start in range(0, len(val_keep), 64):
        rows = val_keep[start:start + 64]
        b = make_batch(val_data, rows, cfg, lists)
        probs = torch.softmax(model(to_device(b)), dim=-1).cpu()
        s = probs.max(dim=-1).values if cfg["score"] == "maxprob" else (probs * torch.arange(4.0)).sum(-1)
        for j, i in enumerate(rows):
            n = int((b["labels"][j] != -1).sum())
            sessions.append(np.full(n, val_data["session_id"][i]))
            labels.append(b["labels"][j, :n].numpy())
            scores.append(s[j, :n].numpy())
    return pd.DataFrame({"session_id": np.concatenate(sessions), "label": np.concatenate(labels),
                         "score": np.concatenate(scores).astype(float)})

Team 3's own evaluation, verbatim, for the sampled validation lists: K = 10 as reported, and K = 5, which their test cell sets in the published notebook (its saved output still reads HR@10 and NDCG@10).

In [7]:
def team3_metrics(frame, K):
    ndcg, hit = [], []
    for _, g in frame.groupby("session_id", sort=False):
        labels, probs = g.label.to_numpy(), g.score.to_numpy()
        top = np.argsort(-probs)[:min(K, len(probs))]
        top_labels = labels[top]
        hit.append(int((top_labels > 0).any()))
        discounts = np.log2(np.arange(2, len(top_labels) + 2))
        dcg = np.sum((2 ** top_labels - 1) / discounts)
        ideal = np.sort(labels)[::-1][:len(top_labels)]
        idcg = np.sum((2 ** ideal - 1) / discounts) if np.sum(2 ** ideal - 1) > 0 else 1.0
        ndcg.append(dcg / idcg)
    return np.mean(ndcg), np.mean(hit)

## 5. Run the ladder

In [8]:
RUNGS = {
    "R0 Team 3 as published": dict(history="filtered", lookup="bug", score="maxprob", train_lists="sampled", extra=False),
    "R0b - label leak (cut from training lists)": dict(history="filtered_train", lookup="bug", score="maxprob", train_lists="sampled", extra=False),
    "R1 + correct candidate lookup": dict(history="filtered_train", lookup="ok", score="maxprob", train_lists="sampled", extra=False),
    "R2 + rank by expected label": dict(history="filtered_train", lookup="ok", score="expected", train_lists="sampled", extra=False),
    "R3 + full history and time gaps": dict(history="full", lookup="ok", score="expected", train_lists="sampled", extra=False),
    "R4 + train on complete lists": dict(history="full", lookup="ok", score="expected", train_lists="full", extra=False),
    "R5 + relevance and popularity": dict(history="full", lookup="ok", score="expected", train_lists="full", extra=True),
}
val_sizes = pd.Series(np.diff(val_data["cand_offsets"])[val_keep], index=val_data["session_id"][val_keep])
per_search, team3_view, timings = {}, {}, {}
for name, cfg in RUNGS.items():
    for seed in SEEDS:
        key = name if seed == SEEDS[0] else f"{name} (seed {seed})"
        print(key)
        start = time.time()
        model = train_rung(cfg, seed)
        full = score_validation(model, cfg, "full")
        per_search[key] = jd.evaluate(full, "score", expected_sizes=val_sizes)
        sampled = score_validation(model, cfg, "sampled")
        team3_view[key] = {"Team 3 NDCG@5": team3_metrics(sampled, 5)[0], "Team 3 NDCG@10": team3_metrics(sampled, 10)[0],
                           "Team 3 HR@10": team3_metrics(sampled, 10)[1]}
        timings[key] = time.time() - start
        print(f"  NDCG@10 (our protocol) {per_search[key].ndcg.mean():.4f}; {timings[key] / 60:.1f} min")
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

R0 Team 3 as published
  epoch 1: train 1.3517, inner val 1.2932, 114 s
  epoch 2: train 1.2930, inner val 1.2843, 113 s
  epoch 3: train 1.2729, inner val 1.2835, 112 s
  epoch 4: train 1.2409, inner val 1.2914, 113 s
  epoch 5: train 1.1771, inner val 1.3545, 112 s
  epoch 6: train 1.0944, inner val 1.4309, 112 s
  NDCG@10 (our protocol) 0.1829; 11.7 min
R0b - label leak (cut from training lists)
  epoch 1: train 1.3528, inner val 1.2937, 110 s
  epoch 2: train 1.2933, inner val 1.2850, 110 s
  epoch 3: train 1.2730, inner val 1.2893, 110 s
  epoch 4: train 1.2411, inner val 1.2981, 110 s
  epoch 5: train 1.1773, inner val 1.3380, 110 s
  NDCG@10 (our protocol) 0.1159; 9.5 min
R1 + correct candidate lookup
  epoch 1: train 1.3778, inner val 1.3324, 111 s
  epoch 2: train 1.3094, inner val 1.3156, 111 s
  epoch 3: train 1.2651, inner val 1.3078, 111 s
  epoch 4: train 1.2041, inner val 1.3337, 111 s
  epoch 5: train 1.1062, inner val 1.4132, 111 s
  epoch 6: train 0.9916, inner val 1.

## 6. Results

Reference points on the same validation searches: random order (all candidates tied) and history popularity alone (the `pop_history` column of the R5 inputs, a monotone transform of the count, so the ranking is identical).

In [9]:
reference = {}
frame = pd.DataFrame({"session_id": np.repeat(val_data["session_id"][val_keep], val_sizes.to_numpy())})
c_rows = np.concatenate([np.arange(*val_data["cand_offsets"][[i, i + 1]]) for i in val_keep])
frame["label"] = val_data["label"][c_rows]
frame["score"] = 0.0
reference["random order"] = jd.evaluate(frame, "score", expected_sizes=val_sizes)
frame["score"] = val_data["r5_features"][c_rows, info["r5_features"].index("pop_history")].astype(float)
reference["history popularity"] = jd.evaluate(frame, "score", expected_sizes=val_sizes)

summary = pd.concat({**{k: jd.summarise(v).iloc[0] for k, v in reference.items()},
                     **{k: jd.summarise(v).iloc[0] for k, v in per_search.items()}}, axis=1).T
summary = summary.join(pd.DataFrame(team3_view).T)
keys = list(per_search)
steps = {keys[0]: jd.paired_difference(per_search[keys[0]], reference["random order"])}
for prev, cur in zip(keys, keys[1:]):
    if "(seed" not in cur and "(seed" not in prev:
        steps[cur] = jd.paired_difference(per_search[cur], per_search[prev])
summary["step_gain"] = pd.Series({k: v[0] for k, v in steps.items()})
summary["step_low"] = pd.Series({k: v[1] for k, v in steps.items()})
summary["step_high"] = pd.Series({k: v[2] for k, v in steps.items()})
summary["minutes"] = pd.Series(timings) / 60
if not SMOKE:
    pd.concat({k: v.ndcg for k, v in {**reference, **per_search}.items()}, axis=1).to_parquet(
        ARTIFACTS / "ladder_per_search.parquet")
    summary.to_csv(ARTIFACTS / "ladder_summary.csv")
    searches = pd.read_parquet(paths.processed / "search_table.parquet")
    for name, per in per_search.items():
        clean = per.index[~searches.final_gap_clamped.reindex(per.index).to_numpy()]
        for split_name, ids in [("validation", per.index), ("validation_clean", clean)]:
            jd.log_result(paths, "04c", "Team 3 re-implementation", f"Team 3 ladder {name}", split_name, per.loc[ids],
                          notes="Team 3's hybrid NCF + LSTM re-implemented; trained on the dev subsample")
summary

,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,Team 3 NDCG@5,Team 3 NDCG@10,Team 3 HR@10,step_gain,step_low,step_high,minutes
random order,19772.0,0.084481,0.083040,0.086023,0.159877,0.101972,NaN,NaN,NaN,NaN,NaN,NaN,NaN
history popularity,19772.0,0.340944,0.336386,0.345528,0.526213,0.336507,NaN,NaN,NaN,NaN,NaN,NaN,NaN
R0 Team 3 as published,19772.0,0.182923,0.178896,0.186875,0.278930,0.201986,0.482309,0.539597,0.983310,0.098442,0.094687,0.102250,11.737157
R0b - label leak (cut from training lists),19772.0,0.115913,0.113086,0.118752,0.198616,0.132410,0.468816,0.527554,0.987002,-0.067010,-0.070816,-0.062794,9.546823
R1 + correct candidate lookup,19772.0,0.158725,0.155489,0.162224,0.278750,0.166707,0.547074,0.598288,0.990643,0.042812,0.039098,0.046454,11.482128
R2 + rank by expected label,19772.0,0.141525,0.138171,0.145018,0.248643,0.151239,0.516876,0.571443,0.984776,-0.017200,-0.020590,-0.014156,11.511274
R3 + full history and time gaps,19772.0,0.141606,0.138333,0.144942,0.247597,0.151684,0.513741,0.568872,0.985232,0.000081,-0.002706,0.002699,16.514907
R4 + train on complete lists,19772.0,0.148466,0.144934,0.151897,0.259353,0.158391,0.526756,0.580370,0.987002,0.006860,0.004790,0.008731,16.785761
R5 + relevance and popularity,19772.0,0.298126,0.293946,0.302565,0.495937,0.289359,0.699689,0.735175,0.997926,0.149660,0.145409,0.154069,14.030535


`step_gain`: paired difference in NDCG@10 to the previous rung (R0: to random order), with a 95% bootstrap interval over searches (training variance across seeds not included). The Team 3 columns use their own protocol on the sampled validation lists, for comparison with their reported HR@10 0.978 and NDCG@10 0.568.

## Summary

Validation: 19,772 searches with an engaged candidate, complete lists, our protocol (the `validation_clean` numbers are in `results.csv`).

- **The rebuild matches Team 3's reported behaviour on their protocol** (R0: HR@10 0.983 against their 0.978, NDCG@10 0.540 against their 0.568), and on that protocol its NDCG@10 of 0.540 is **about that of random order** (0.532, notebook 04).
- **On our protocol R0 reaches 0.183, and about two thirds of its gain over random order (0.084) is the label leak** through their catalogue cut: building the cut from training lists only (R0b) costs 0.067 [0.063, 0.071] of the 0.098 gain.
- **The code and design fixes add little** on their architecture: correct lookup +0.043 [0.039, 0.046], complete training lists +0.007 [0.005, 0.009], the full history +0.000 [-0.003, 0.003]; ranking by the expected label costs 0.017 [0.014, 0.021] (inverse-frequency class weights inflate cart and purchase probabilities, so the expected label is noisy; untested hypothesis).
- **Adding relevance and popularity (R5) helps most (+0.150 [0.145, 0.154], to 0.298), but the model still ranks below sorting by popularity alone** (history popularity 0.341): a per-candidate classifier over summed ID embeddings, trained with cross-entropy, does not exploit even the strongest single signal.
- All rungs overfit early (best epoch 2-3 of at most 10), as in Team 3's own runs. One seed per rung.
- **Consequence:** Team 3's architecture is not a viable base; our models are compared with popularity and a trained non-personalised ranker (notebook 05).